In [0]:
dbutils.widgets.text("batch_id","")
v_batch_id = dbutils.widgets.get("batch_id")

In [0]:
%run ../0-common/env-config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_table"

In [0]:
results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
    .filter(F.col("batch_id") == v_batch_id)
    .withColumn("session_type", F.lit("RACE"))
    .drop("race_date", "race_name", "ingestion_timestamp", "source_file","batch_id","created_at","updated_at")
)

In [0]:
sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
    .filter(F.col("batch_id") == v_batch_id)
    .withColumn("session_type", F.lit("SPRINT"))
    .drop("race_date", "race_name", "ingestion_timestamp", "source_file","batch_id","created_at","updated_at")
)

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

In [0]:
fact_session_result_df =  (
    results_sprints_df
    .withColumn("is_win", F.col("final_position") == 1)
    .withColumn("is_podium", F.col("final_position").between(1,3))
    .withColumn("has_points", F.col("points") > 0)
    .withColumn("created_at", F.current_timestamp())
    .withColumn("updated_at", F.current_timestamp())
)

In [0]:
if not spark.catalog.tableExists(target_table):
    (
        fact_session_result_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )
else:
    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, target_table)
    (
        delta_table.alias("t")
        .merge(
            fact_session_result_df.alias("c"),
            """t.season = c.season
            AND t.round = c.round
            AND t.constructor_id = c.constructor_id
            AND t.driver_id = c.driver_id 
            AND t.session_type = c.session_type"""
        )
        .whenMatchedUpdate(
            set={
                "grid_position": "c.grid_position",
                "laps": "c.laps",
                "car_number": "c.car_number",
                "points": "c.points",
                "final_position": "c.final_position",
                "final_position_text": "c.final_position_text",
                "status": "c.status",
                "is_win": "c.is_win",
                "is_podium": "c.is_podium",
                "has_points": "c.has_points",
                "updated_at": "c.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )